In [2]:
import xarray as xr

ds_hgt_pwat = xr.open_dataset("/home/scratch/dwefer/GEFSv12/z500_pwat.nc")
ds_cape = xr.open_dataset("/home/scratch/dwefer/GEFSv12/cape.nc")

ds_cape

<xarray.Dataset> Size: 2GB
Dimensions:     (time: 7305, latitude: 159, longitude: 359)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 58kB 2000-01-01 2000-01-02 ... 2019-12-31
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 1kB 55.0 54.75 54.5 54.25 ... 16.0 15.75 15.5
    valid_time  (time) datetime64[ns] 58kB ...
  * longitude   (longitude) float64 3kB -140.0 -139.8 -139.5 ... -50.75 -50.5
Data variables:
    cape        (time, latitude, longitude) float32 2GB ...

In [4]:
print("cape lat:", ds_cape.latitude.size, ds_cape.latitude.values[:3], ds_cape.latitude.values[-3:])
print("z500 + pwat  lat:", ds_hgt_pwat.latitude.size,  ds_hgt_pwat.latitude.values[:3], ds_hgt_pwat.latitude.values[-3:])
print("cape lon:", ds_cape.longitude.size, ds_cape.longitude.values[:3], ds_cape.longitude.values[-3:])
print("z500 + pwat  lon:", ds_hgt_pwat.longitude.size,  ds_hgt_pwat.longitude.values[:3],  ds_hgt_pwat.longitude.values[-3:])

cape lat: 159 [55.   54.75 54.5 ] [16.   15.75 15.5 ]
z500 + pwat  lat: 80 [55.  54.5 54. ] [16.5 16.  15.5]
cape lon: 359 [-140.   -139.75 -139.5 ] [-51.   -50.75 -50.5 ]
z500 + pwat  lon: 180 [220.  220.5 221. ] [308.5 309.  309.5]


In [5]:
ds_cape = ds_cape.assign_coords(longitude=(ds_cape.longitude % 360)).sortby("longitude")

## pwat is at a 0.25 degree res, z500 is at a 0.5 degree res. Going to coarsen to input into the som

In [6]:
ds_cape = ds_cape.sortby("time")
ds_hgt_pwat  = ds_hgt_pwat.sortby("time")

ds_coarse_cape = ds_cape.reindex(
    latitude=ds_hgt_pwat.latitude,
    longitude=ds_hgt_pwat.longitude,
    method="nearest"
)

ds_out = xr.merge([ds_coarse_cape, ds_hgt_pwat], join="exact")

ds_out.to_netcdf("/home/scratch/dwefer/GEFSv12/z500_pwat_cape.nc")